In [1]:
import duckdb
import pandas as pd

# Connect DuckDB to the parquet file
con = duckdb.connect()

# Test the connection
result = con.execute("""
    SELECT COUNT(*) as total_loans,
           ROUND(AVG(TARGET)*100, 2) as default_rate_pct,
           COUNT(DISTINCT grade) as grades
    FROM read_parquet('E:/finsight/data/processed/loans_clean.parquet')
""").df()

print(result)

   total_loans  default_rate_pct  grades
0      1328284             21.41       7


In [3]:
# ── Q1: Portfolio quality by grade ─────────────────────────────────
print("=" * 60)
print("Q1: PORTFOLIO QUALITY BY GRADE")
print("=" * 60)
q1 = con.execute("""
    SELECT
        grade,
        COUNT(*) AS loan_count,
        ROUND(AVG(TARGET)*100, 2) AS default_rate_pct,
        ROUND(AVG(int_rate), 2) AS avg_interest_rate,
        ROUND(AVG(loan_amnt), 0) AS avg_loan_amt,
        ROUND(SUM(loan_amnt * TARGET)/1e6, 2) AS default_exposure_millions,
        ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER(), 1) AS pct_of_portfolio
    FROM read_parquet('E:/finsight/data/processed/loans_clean.parquet')
    GROUP BY grade
    ORDER BY grade
""").df()
print(q1.to_string(index=False))

# ── Q2: Default rate by loan purpose ───────────────────────────────
print("\n" + "=" * 60)
print("Q2: DEFAULT RATE BY LOAN PURPOSE")
print("=" * 60)
q2 = con.execute("""
    SELECT
        purpose,
        COUNT(*) AS loan_count,
        ROUND(AVG(loan_amnt), 0) AS avg_loan_amt,
        ROUND(AVG(TARGET)*100, 2) AS default_rate_pct,
        ROUND(AVG(int_rate), 2) AS avg_rate,
        ROUND(AVG(dti), 2) AS avg_dti
    FROM read_parquet('E:/finsight/data/processed/loans_clean.parquet')
    GROUP BY purpose
    ORDER BY default_rate_pct DESC
""").df()
print(q2.to_string(index=False))

# ── Q3: Risk tier by DTI bucket ─────────────────────────────────────
print("\n" + "=" * 60)
print("Q3: RISK TIER BY DTI BUCKET")
print("=" * 60)
q3 = con.execute("""
    SELECT
        CASE
            WHEN dti < 10  THEN 'Low DTI (<10%)'
            WHEN dti < 20  THEN 'Moderate (10-20%)'
            WHEN dti < 30  THEN 'Elevated (20-30%)'
            ELSE 'High DTI (>30%)'
        END AS dti_tier,
        COUNT(*) AS count,
        ROUND(AVG(TARGET)*100, 2) AS default_rate_pct,
        ROUND(AVG(int_rate), 2) AS avg_rate,
        ROUND(AVG(loan_amnt), 0) AS avg_loan_amt
    FROM read_parquet('E:/finsight/data/processed/loans_clean.parquet')
    GROUP BY dti_tier
    ORDER BY default_rate_pct DESC
""").df()
print(q3.to_string(index=False))

# ── Q4: Loan volume + default rate by year ──────────────────────────
print("\n" + "=" * 60)
print("Q4: PORTFOLIO TREND BY YEAR")
print("=" * 60)
q4 = con.execute("""
    SELECT
        issue_year,
        COUNT(*) AS loans_issued,
        ROUND(SUM(loan_amnt)/1e6, 1) AS volume_millions,
        ROUND(AVG(TARGET)*100, 2) AS default_rate_pct,
        ROUND(AVG(int_rate), 2) AS avg_rate
    FROM read_parquet('E:/finsight/data/processed/loans_clean.parquet')
    WHERE issue_year IS NOT NULL
    GROUP BY issue_year
    ORDER BY issue_year
""").df()
print(q4.to_string(index=False))

# ── Q5 Fixed: Income quintile analysis ─────────────────────────────
print("=" * 60)
print("Q5: DEFAULT RATE BY INCOME QUINTILE")
print("=" * 60)
q5 = con.execute("""
    WITH quintiles AS (
        SELECT *,
            NTILE(5) OVER (ORDER BY annual_inc) AS income_quintile
        FROM read_parquet('E:/finsight/data/processed/loans_clean.parquet')
    )
    SELECT
        income_quintile,
        ROUND(MIN(annual_inc), 0) AS min_income,
        ROUND(MAX(annual_inc), 0) AS max_income,
        COUNT(*) AS count,
        ROUND(AVG(TARGET)*100, 2) AS default_rate_pct,
        ROUND(AVG(dti), 2) AS avg_dti,
        ROUND(AVG(loan_amnt), 0) AS avg_loan_amt
    FROM quintiles
    GROUP BY income_quintile
    ORDER BY income_quintile
""").df()
print(q5.to_string(index=False))

# ── Q6: Top 3 sub-grades by default rate per grade ──────────────────
print("\n" + "=" * 60)
print("Q6: TOP 3 RISKIEST SUB-GRADES PER GRADE")
print("=" * 60)
q6 = con.execute("""
    SELECT * FROM (
        SELECT
            grade, sub_grade,
            COUNT(*) AS count,
            ROUND(AVG(TARGET)*100, 2) AS default_rate_pct,
            ROUND(AVG(int_rate), 2) AS avg_rate,
            RANK() OVER (PARTITION BY grade ORDER BY AVG(TARGET) DESC) AS rnk
        FROM read_parquet('E:/finsight/data/processed/loans_clean.parquet')
        GROUP BY grade, sub_grade
        HAVING COUNT(*) > 100
    ) t WHERE rnk <= 3
    ORDER BY grade, rnk
""").df()
print(q6.to_string(index=False))

# ── Q7: Employment length vs default rate ───────────────────────────
print("\n" + "=" * 60)
print("Q7: DEFAULT RATE BY EMPLOYMENT LENGTH")
print("=" * 60)
q7 = con.execute("""
    SELECT
        emp_length,
        COUNT(*) AS count,
        ROUND(AVG(TARGET)*100, 2) AS default_rate_pct,
        ROUND(AVG(annual_inc), 0) AS avg_income,
        ROUND(AVG(loan_amnt), 0) AS avg_loan
    FROM read_parquet('E:/finsight/data/processed/loans_clean.parquet')
    GROUP BY emp_length
    ORDER BY default_rate_pct DESC
""").df()
print(q7.to_string(index=False))

# ── Q8: Home ownership vs default ───────────────────────────────────
print("\n" + "=" * 60)
print("Q8: DEFAULT RATE BY HOME OWNERSHIP")
print("=" * 60)
q8 = con.execute("""
    SELECT
        home_ownership,
        COUNT(*) AS count,
        ROUND(AVG(TARGET)*100, 2) AS default_rate_pct,
        ROUND(AVG(loan_amnt), 0) AS avg_loan,
        ROUND(AVG(annual_inc), 0) AS avg_income,
        ROUND(AVG(dti), 2) AS avg_dti
    FROM read_parquet('E:/finsight/data/processed/loans_clean.parquet')
    GROUP BY home_ownership
    ORDER BY default_rate_pct DESC
""").df()
print(q8.to_string(index=False))

# ── Save all outputs ─────────────────────────────────────────────────
q1.to_csv(r'E:\finsight\data\outputs\q1_grade_analysis.csv', index=False)
q2.to_csv(r'E:\finsight\data\outputs\q2_purpose_analysis.csv', index=False)
q3.to_csv(r'E:\finsight\data\outputs\q3_dti_analysis.csv', index=False)
q4.to_csv(r'E:\finsight\data\outputs\q4_yearly_trend.csv', index=False)
q5.to_csv(r'E:\finsight\data\outputs\q5_income_quintile.csv', index=False)
q6.to_csv(r'E:\finsight\data\outputs\q6_subgrade_risk.csv', index=False)
q7.to_csv(r'E:\finsight\data\outputs\q7_employment.csv', index=False)
q8.to_csv(r'E:\finsight\data\outputs\q8_home_ownership.csv', index=False)

print("\n✓ All outputs saved to data/outputs/")

Q1: PORTFOLIO QUALITY BY GRADE
grade  loan_count  default_rate_pct  avg_interest_rate  avg_loan_amt  default_exposure_millions  pct_of_portfolio
    A      227657              6.63               7.13       13881.0                     208.96              17.1
    B      385270             14.52              10.69       13250.0                     765.34              29.0
    C      378206             24.08              14.02       14218.0                    1353.69              28.5
    D      200964             32.16              17.72       15284.0                    1037.34              15.1
    E       94336             40.02              21.13       17601.0                     683.73               7.1
    F       32450             46.46              24.91       19074.0                     295.10               2.4
    G        9401             51.16              27.57       20440.0                      98.06               0.7

Q2: DEFAULT RATE BY LOAN PURPOSE
           purpose  loa